<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Two signal checks (before I trust my rule)
**Signal 1 — CTR vs. position** (flag-linked: this is the assumption behind the CTR-fix flag,
which uses a fixed 0.5% CTR cutoff for pages in position 1–20).

Verdict: CONFIRMED. Mean CTR drops cleanly and monotonically as position worsens —
top_3 (1.06%, n=16,144) > page_1 (0.49%, n=81,988) > striking (0.32%, n=32,203) >
page_3_5 (0.23%, n=33,288) > deep (0.09%, n=11,681). This supports the CTR-fix flag's core
assumption that position and CTR move together. Worth noting: the flag's 0.5% threshold sits
almost exactly at the page_1 bucket's own average — meaning roughly half of position-1-10 pages
already fall under that "low CTR" bar on their own.

**Signal 2 — Volume vs. clicks/CTR** (flag-linked: the assumption behind quick-win logic —
bigger visibility means bigger recoverable opportunity, not that high volume already converts
efficiently on its own).

Verdict: MIXED. Mean clicks scale up with volume as expected — none (0) → low (0.17) →
moderate (3.05) → good (23.46) → excellent (152.38), n ranging 962–154,699 — confirming real
opportunity size exists at every non-trivial tier. But mean CTR does NOT rise with volume: it
falls from low (0.60%) to moderate (0.26%) and stays roughly flat through good (0.30%) and
excellent (0.28%), never recovering. So the "bigger volume = bigger opportunity" half of the
quick-win logic holds, but a naive "more volume = more efficient" reading is false — high-volume
pages are not already converting better, which is exactly why they're worth reviewing.

In [2]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Build the same monthly page-level table as Week 3
monthly = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_month,
        AVG(gsc_avg_position) AS avg_position_month
    FROM {DAILY}
    GROUP BY content_hash_id, client_hash_id
""").df()

print("Rows in monthly table:", len(monthly))

# ---- Signal check 1: CTR vs position (flag-linked -> low_ctr_visible_page / CTR-fix logic) ----
# Only look at pages with a real position value (0 = "no data", not rank zero)
has_position = monthly[monthly["avg_position_month"] > 0].copy()

def position_bucket(p):
    if p <= 3: return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"

has_position["position_bucket"] = has_position["avg_position_month"].apply(position_bucket)

signal1_table = (
    has_position.groupby("position_bucket")
    .agg(n=("ctr_month", "size"), mean_ctr=("ctr_month", "mean"))
    .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
)
print("\n--- Signal 1: CTR by position bucket ---")
print(signal1_table)

# ---- Signal check 2: volume (flag-linked -> quick-win logic) ----
def impression_bucket(i):
    if i == 0: return "none"
    if i < 300: return "low"
    if i < 3000: return "moderate"
    if i < 30000: return "good"
    return "excellent"

monthly["impression_bucket"] = monthly["impressions_month"].apply(impression_bucket)

signal2_table = (
    monthly.groupby("impression_bucket")
    .agg(n=("clicks_month", "size"), mean_clicks=("clicks_month", "mean"), mean_ctr=("ctr_month", "mean"))
    .reindex(["none", "low", "moderate", "good", "excellent"])
)
print("\n--- Signal 2: clicks/CTR by impression volume bucket ---")
print(signal2_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in monthly table: 331437

--- Signal 1: CTR by position bucket ---
                     n  mean_ctr
position_bucket                 
top_3            16144  1.058944
page_1           81988  0.492591
striking         32203  0.321080
page_3_5         33288  0.228706
deep             11681  0.090304

--- Signal 2: clicks/CTR by impression volume bucket ---
                        n  mean_clicks  mean_ctr
impression_bucket                               
none               154699     0.000000       NaN
low                102072     0.172290  0.595242
moderate            52509     3.053515  0.261651
good                21195    23.463930  0.303213
excellent             962   152.381497  0.277453


## 2. Build the ranked queue (writes the CSV)


Rule: A page is worth reviewing first if it already earns real search visibility
(impressions_month >= 500) and holds a real, working position (0 < avg_position_month <= 20),
but converts that visibility into clicks far below the 0.5% CTR benchmark used in signal 1.
The bigger the visibility and the bigger the CTR shortfall below 0.5%, the higher the page ranks.
Reason code: low_ctr_visible_page. Action label: review_ctr_fix if flagged, else monitor.

In [6]:
CTR_BENCHMARK = 0.5  # percent, from signal 1's flag-linked threshold

eligible = (
    (monthly["impressions_month"] >= 500)
    & (monthly["avg_position_month"] > 0)
    & (monthly["avg_position_month"] <= 20)
)

ctr_gap = (CTR_BENCHMARK - monthly["ctr_month"]).clip(lower=0)

monthly["score"] = 0.0
monthly.loc[eligible, "score"] = monthly.loc[eligible, "impressions_month"] * ctr_gap[eligible]

monthly["reason_code"] = "none"
monthly.loc[eligible & (ctr_gap > 0), "reason_code"] = "low_ctr_visible_page"

monthly["action"] = "monitor"
monthly.loc[eligible & (ctr_gap > 0), "action"] = "review_ctr_fix"

queue = monthly.sort_values("score", ascending=False).reset_index(drop=True)

print("Total pages:", len(queue))
print("Flagged for review:", (queue["action"] == "review_ctr_fix").sum())
print("\nTop 10 rows:")
print(queue[["content_hash_id", "impressions_month", "ctr_month", "avg_position_month",
             "score", "reason_code", "action"]].head(10))

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
queue = queue.drop(columns=["impression_bucket"])
print(queue[["content_hash_id", "impressions_month", "ctr_month", "avg_position_month",
             "score", "reason_code", "action"]].iloc[10:20])
print("\nSaved to work/outputs/baseline_action_score.csv")


Total pages: 331437
Flagged for review: 41008

Top 10 rows:
            content_hash_id  impressions_month  ctr_month  avg_position_month  \
0  content_44f34c0a90047651           212404.0       0.01            7.346909   
1  content_8d7d99f109e19aa2           203497.0       0.14            2.563756   
2  content_8e1334d6356668e3           134984.0       0.00            4.545582   
3  content_34a70fea29d15f24           143019.0       0.03            3.219473   
4  content_fec55986a1868d62           124075.0       0.00            9.385150   
5  content_b99ea6861864dea5           194337.0       0.19            4.450106   
6  content_acbcc847f8996314           170808.0       0.15            3.361195   
7  content_7c6373141eae744a           132593.0       0.06            5.789019   
8  content_e8a52cf3d5988c07           244931.0       0.27           15.008339   
9  content_f6116743b00afc2d           107584.0       0.01            9.536301   

       score           reason_code          acti

## 3. Top-20 review

1. content_44f34c0a90047651 — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: High. Why: 212,404 impressions/month at position ~7.3, but CTR of 0.01% —
   essentially zero clicks despite heavy visibility.
   Would be wrong if: the page is a duplicate/redirect target, or impressions are inflated by
   a broad, low-intent query mix that wouldn't convert even with a perfect title/snippet fix.

2. content_8d7d99f109e19aa2 — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: High. Why: 203,497 impressions at a strong position (~2.6) yet only 0.14% CTR —
   the CTR gap is the whole story here, position is fine.
   Would be wrong if: this is a featured-snippet-only ranking that suppresses clicks by design
   (answer shown on the SERP itself), which no title/meta rewrite would fix.

3. content_8e1334d6356668e3 — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: Medium. Why: 134,984 impressions, position ~4.5, CTR of 0.00% — zero measurable
   clicks at a genuinely good position.
   Would be wrong if: this page has a tracking/measurement gap (clicks not firing to GSC)
   rather than a real content/snippet problem — worth a tagging check before a content review.

4. content_34a70fea29d15f24 — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: High. Why: 143,019 impressions at position ~3.2 with 0.03% CTR — near-top
   position, almost no clicks.
   Would be wrong if: the query mix behind these impressions is mostly navigational/branded
   traffic already going elsewhere, which isn't fixable by better copy.

5. content_fec55986a1868d62 — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: Medium. Why: 124,075 impressions, position ~9.4, 0.00% CTR — visible but
   converting nothing.
   Would be wrong if: it's a thin/placeholder page that impresses on a broad keyword it
   shouldn't rank for at all — the fix would be de-indexing or merging, not a CTR tweak.

6. content_b99ea6861864dea5 — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: High. Why: 194,337 impressions at position ~4.5 with 0.19% CTR — high volume,
   position is fine, conversion is the gap.
   Would be wrong if: the page recently changed URL/title and GSC data blends old and new
   metadata performance, making the CTR figure temporarily unreliable.

7. content_acbcc847f8996314 — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: High. Why: 170,808 impressions, position ~3.4, CTR 0.15% — another
   strong-position, weak-conversion case.
   Would be wrong if: this page competes against a very similar sibling page from the same
   client that's stealing the clicks — the fix might be consolidation, not a rewrite.

8. content_7c6373141eae744a — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: Medium. Why: 132,593 impressions at position ~5.8, CTR 0.06% — solid position,
   clicks not following.
   Would be wrong if: seasonal/trend-driven impressions spiked briefly this month and the CTR
   figure reflects an unrepresentative short window rather than a stable pattern.

9. content_e8a52cf3d5988c07 — action: review_ctr_fix, reason: low_ctr_visible_page.
   Confidence: Low. Why: 244,931 impressions (largest in the top 10) at position ~15, CTR
   0.27% — biggest raw opportunity by volume, even though the CTR gap itself is the mildest
   in this list.
   Would be wrong if: position ~15 is genuinely borderline for any click-through regardless of
   snippet quality — the real lever here might be a ranking/position fix, not a CTR fix.

10. content_f6116743b00afc2d — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: Medium. Why: 107,584 impressions at position ~9.5, CTR 0.01% — smallest volume
    in the top 10 but still large in absolute terms, near-zero conversion.
    Would be wrong if: this is a non-intent-matching ranking (e.g. ranking for a question the
    page doesn't actually answer), where the real fix is content relevance, not snippet/CTR
    optimization.

11. content_f43118e089ecc69a — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: High. Why: 139,417 impressions at position ~5.0, CTR 0.14% — good position,
    conversion far behind what it should support.
    Would be wrong if: this page's traffic is dominated by a single spiky, low-intent query
    that inflates impressions without representing genuine demand.

12. content_cd3d932d4e1c8db0 — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: Medium. Why: 89,332 impressions, position ~7.8, CTR 0.00% — smallest volume so
    far in the list, but still zero measurable clicks.
    Would be wrong if: this is a tracking/measurement gap rather than a real content issue —
    worth confirming GSC click tracking is firing correctly before a content review.

13. content_5e1c049f62e33b11 — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: Low. Why: 120,175 impressions, CTR 0.14%, but position ~18.1 — the weakest
    position in the top 20, right at the edge of the eligibility window (<=20).
    Would be wrong if: position ~18 is simply too far down the results page for any snippet
    fix to matter — the real lever is a ranking improvement, not a CTR-focused edit.

14. content_471d9cabce329a66 — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: High. Why: 164,885 impressions at a strong position (~4.7) with CTR 0.24% —
    high volume and a real, if smaller, CTR gap.
    Would be wrong if: this page recently underwent a title/meta change and the CTR figure
    is still stabilizing, mixing old and new performance in one month's average.

15. content_e578ac84778da489 — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: High. Why: 117,764 impressions at position ~4.1 with CTR 0.14% — strong
    position, meaningful shortfall.
    Would be wrong if: this page ranks for a mostly informational/no-click-intent query
    (e.g. answered directly on the SERP) where low CTR is structural, not fixable.

16. content_9c057b66c30a3abb — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: Medium. Why: 83,834 impressions, position ~11.2, CTR 0.00% — lower volume,
    but a complete conversion failure at a workable position.
    Would be wrong if: this page duplicates content already ranking elsewhere for the same
    client, and the "fix" is really consolidation, not a rewrite.

17. content_046fc480045b88f5 — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: Medium. Why: 83,788 impressions at position ~7.3, CTR 0.01% — near-zero clicks
    at a decent position.
    Would be wrong if: this is a seasonal or short-lived spike in impressions and the CTR
    figure doesn't reflect a stable monthly pattern.

18. content_9540d884af3e41fd — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: Medium. Why: 82,376 impressions, position ~7.8, CTR 0.01% — smallest raw
    volume in the top 20, still a clear shortfall.
    Would be wrong if: the query mix behind these impressions is largely irrelevant/mismatched
    to the page's actual content, meaning the fix is relevance, not presentation.

19. content_f352b7cfd0b2f434 — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: High. Why: 136,098 impressions at a very strong position (~3.3) with CTR
    0.21% — one of the best positions in the top 20, still underconverting.
    Would be wrong if: this page competes with a sibling page from the same client for the
    same query, splitting clicks that would otherwise land here.

20. content_77276ad7a26f4905 — action: review_ctr_fix, reason: low_ctr_visible_page.
    Confidence: High. Why: 116,707 impressions at position ~3.9 with CTR 0.17% — strong
    position, real gap.
    Would be wrong if: recent algorithm volatility temporarily boosted this page's position
    without genuine ranking improvement, making the current position/CTR combo unstable.    

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
used_cols = {"content_hash_id", "client_hash_id", "impressions_month", "clicks_month",
             "ctr_month", "avg_position_month", "score", "reason_code", "action"}
print("Columns in final queue:", set(queue.columns))
print("Any unexpected columns beyond the honest feature set:",
      set(queue.columns) - used_cols)

Columns in final queue: {'score', 'content_hash_id', 'avg_position_month', 'client_hash_id', 'reason_code', 'ctr_month', 'impressions_month', 'action', 'clicks_month'}
Any unexpected columns beyond the honest feature set: set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.